# Day 11–12 — Channel Reduction Study  
## Structural Sensitivity of CSP Under Electrode Constraints

### Objective
Evaluate how channel count influences CSP performance relative to baseline under simulated consumer-grade constraints.

Regularized CSP was selected to improve robustness under:
- Limited data
- Noisy channels
- Reduced electrode counts

Noise levels evaluated:
- STD = 0 (clean)
- STD = 1.0 (noisy)

---

## Core Observations

### 1. High Channel Counts (13–16)
- Baseline accuracy exceeds CSP.
- ΔAccuracy (CSP − Baseline) becomes negative at 16 channels.
- Additional electrodes likely introduce covariance noise that weakens spatial filtering.

### 2. Mid-Range Channels (8–12)
- Reducing channels from 13 to 8 produces positive ΔAccuracy.
- Largest CSP gain occurs at 8 channels (~+0.0175).
- 8-channel CSP outperforms 16-channel baseline by ~0.025 accuracy.
- This configuration represents the strongest CSP dominance regime.

### 3. Low Channel Counts (2–4)
- ΔAccuracy declines toward zero.
- Insufficient spatial resolution limits discriminative filtering.

---

## Noise Effects
- Accuracy remains largely stable as noise STD increases to 1.0.
- Channel configuration influences performance more strongly than noise magnitude.
- Standard deviations remain mostly constant across noise levels.
- Channel reduction shifts accuracy curves vertically without materially altering variability.

---

## Identified Pre-Lock Peak Configuration
Most effective electrode set:
F4, FC2, FC1, FC5, FC6, Cz, C3, C4

---

## Structural Interpretation
- Channel count is the dominant determinant of performance.
- CSP dominates roughly within the 2–12 channel range.
- Baseline surpasses CSP beyond ~13 channels.
- Excess channels may dilute covariance structure rather than enhance it.
- Accuracy differences are primarily channel-dependent, not noise-dependent.

This stage maps the structural sensitivity regime prior to final configuration lock.

In [ ]:
# =========================
# DAY 11 — Channel Reduction + CSP Collapse Test
# =========================

!pip install mne
import numpy as np
np.random.seed(42)
import pandas as pd
from google.colab import drive
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from mne.decoding import CSP

# -------------------------
# Mount Drive
# -------------------------
drive.mount("/content/drive")

# -------------------------
# Load pre-CSP epoched data
# -------------------------
X = np.load("/content/drive/MyDrive/EEG/epochs_day7.npy")   # (trials, channels, samples)
y = np.load("/content/drive/MyDrive/EEG/labels_day7.npy")

print("Epoch data:", X.shape)
print("Labels:", y.shape)

# -------------------------
# Channel names (MUST match data)
# -------------------------
CHANNELS = [
    "T7", "T8", "Fz", "CP2", "CP1", "CP5", "CP6", "F3", "F4", "FC2","FC1", "FC5", "FC6", "Cz", "C3", "C4"
]

assert X.shape[1] == len(CHANNELS), "Channel count mismatch"

# -------------------------
# Channel reduction sets
# -------------------------
# -------------------------
# Symmetric Channel Reduction Ladder
# -------------------------

channel_sets = {

    # 16 (full)
    16: ["T7", "T8",
         "Fz",
         "CP2", "CP1",
         "CP5", "CP6",
         "F3", "F4",
         "FC2", "FC1",
         "FC5", "FC6",
         "Cz",
         "C3", "C4"],

    # 14 (remove outer temporal pair T7/T8)
    14: ["Fz",
         "CP2", "CP1",
         "CP5", "CP6",
         "F3", "F4",
         "FC2", "FC1",
         "FC5", "FC6",
         "Cz",
         "C3", "C4"],

    # 12 (remove CP5/CP6)
    12: ["Fz",
         "CP2", "CP1",
         "F3", "F4",
         "FC2", "FC1",
         "FC5", "FC6",
         "Cz",
         "C3", "C4"],

    # 10 (remove CP1/CP2)
    10: ["Fz",
         "F3", "F4",
         "FC2", "FC1",
         "FC5", "FC6",
         "Cz",
         "C3", "C4"],

    # 8 (remove F3/F4)
    8: ["Fz",
        "FC2", "FC1",
        "FC5", "FC6",
        "Cz",
        "C3", "C4"],

    # 6 (remove FC2/FC1)
    6: ["Fz",
        "FC5", "FC6",
        "Cz",
        "C3", "C4"],

    # 5 (remove Fz → now odd but still symmetric around midline)
    5: ["FC5", "FC6",
        "Cz",
        "C3", "C4"],

    # 3 (remove FC5/FC6)
    3: ["Cz",
        "C3", "C4"],

    # 2 (minimal motor pair)
    2: ["C3", "C4"]
}

# -------------------------
# Noise injection (sensor noise model)
# -------------------------
def inject_noise(X, std):
    return X + np.random.normal(0, std, X.shape)

# -------------------------
# Experiment parameters
# -------------------------
N_COMPONENTS = 6
NOISE_LEVELS = np.linspace(0, 1.0, 6)  # 0.0 → 1.0
N_FOLDS = 6

results = []

# -------------------------
# Main experiment loop
# -------------------------
for n_ch, ch_names in channel_sets.items():

    ch_idx = [CHANNELS.index(ch) for ch in ch_names]
    X_sub = X[:, ch_idx, :]

    print(f"\nChannels: {n_ch} → {ch_names}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    for noise_std in NOISE_LEVELS:

        fold_acc_csp = []
        fold_acc_base = []

        for train_idx, test_idx in skf.split(X_sub, y):

            X_tr, X_te = X_sub[train_idx], X_sub[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            # -------- CSP pipeline --------
            csp = CSP(n_components=min(N_COMPONENTS, n_ch), log=True, norm_trace=False)
            X_tr_csp = csp.fit_transform(X_tr, y_tr)
            X_te_csp = csp.transform(inject_noise(X_te, noise_std))

            lda = LinearDiscriminantAnalysis()
            lda.fit(X_tr_csp, y_tr)
            fold_acc_csp.append(accuracy_score(y_te, lda.predict(X_te_csp)))

            # -------- Baseline (bandpower-like) --------
            X_tr_base = X_tr.var(axis=2)
            X_te_base = inject_noise(X_te, noise_std).var(axis=2)

            lda_base = LinearDiscriminantAnalysis()
            lda_base.fit(X_tr_base, y_tr)
            fold_acc_base.append(accuracy_score(y_te, lda_base.predict(X_te_base)))

        results.append({
            "channels": n_ch,
            "noise_std": noise_std,
            "csp_mean_acc": np.mean(fold_acc_csp),
            "csp_std": np.std(fold_acc_csp),
            "base_mean_acc": np.mean(fold_acc_base),
            "base_std": np.std(fold_acc_base),
        })

# -------------------------
# Save results
# -------------------------
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/EEG/day11_channel_reduction_results.csv", index=False)

print("\nDay 11 results saved.")
